# 1 — Heterogeneity: what the clients actually see

Two axes are injected independently:

- **feature shift** — each client is assigned one rotation group (0/90/180/270°), applied losslessly with `np.rot90`
- **label skew** — a Dirichlet(α) split of the class indices across clients

This notebook builds the benchmark and shows both axes. Runs in under a minute, CPU only.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # import hefl from the repo root

import numpy as np, torch, matplotlib.pyplot as plt
from hefl.utils import pick_device, set_seed
set_seed(42); DEVICE = pick_device('auto')
print('device:', DEVICE)


In [ ]:
from hefl.datasets import build_federated_data

data = build_federated_data(
    root='../data_cache', num_clients=24, alpha=0.5,
    rotation_groups=[0, 1, 2, 3], seed=42,
)
print(f'clients          : {data.num_clients}')
print(f'samples/client   : min {min(data.client_sizes())}, max {max(data.client_sizes())}')
print(f'test sets        : {len(data.test_sets)} (one per rotation group, {len(data.test_sets[0])} images each)')


## The feature axis

One image from a client in each rotation group. The rotation is a property of the *client*, so every sample that client holds carries it.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
mean = np.array([0.4914, 0.4822, 0.4465]); std = np.array([0.2470, 0.2435, 0.2616])
for g in range(4):
    cid = int(np.where(data.client_rotation == g)[0][0])
    img = data.client_sets[cid][0][0].permute(1, 2, 0).numpy() * std + mean
    axes[g].imshow(np.clip(img, 0, 1)); axes[g].set_title(f'group {g} — {g*90}°'); axes[g].axis('off')
fig.suptitle('Feature axis: one client per rotation group', y=1.04); plt.tight_layout(); plt.show()


## The label axis

Each column is a client, each colour a class. With α = 0.5 the columns are visibly uneven — that is the label skew.


In [ ]:
hist = data.client_label_hist
fig, ax = plt.subplots(figsize=(11, 3.5))
bottom = np.zeros(len(hist))
for c in range(10):
    ax.bar(range(len(hist)), hist[:, c], bottom=bottom, width=0.85,
           color=plt.cm.tab10(c), label=f'{c}' if len(hist) < 30 else None)
    bottom += hist[:, c]
ax.set_xlabel('client'); ax.set_ylabel('class proportion')
ax.set_title(f'Label axis: Dirichlet(α=0.5) across {len(hist)} clients')
ax.legend(ncol=10, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.22))
plt.tight_layout(); plt.show()


## The two axes are independent

Rotation group tells you nothing about label distribution, which is exactly the point: a clustering signal that recovers one must not be a proxy for the other.


In [ ]:
from scipy.stats import entropy
for g in range(4):
    members = np.where(data.client_rotation == g)[0]
    ent = [entropy(hist[i]) for i in members]
    print(f'group {g} ({g*90:>3}°): {len(members)} clients, mean label entropy {np.mean(ent):.3f} bits')
print('\nSimilar entropy across groups => the two axes are not correlated by construction.')
